# Boston Housing - EDA et Prédiction de MEDV avec XGBoost

Notebook complet : exploration des données, statistiques descriptives, visualisations, corrélations, entraînement et évaluation du modèle XGBoost.

In [1]:
pip install pandas numpy scikit-learn xgboost matplotlib seaborn

     ---------------------------------------- 0.0/294.9 kB ? eta -:--:--
     ------------------------------------ - 286.7/294.9 kB 6.0 MB/s eta 0:00:01
     -------------------------------------- 294.9/294.9 kB 3.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

sns.set_style('whitegrid')

In [ ]:
# Chargement des données
DATA_URL='https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv'
housing_data=pd.read_csv(DATA_URL)
print(housing_data.head())
print('Dimensions :', housing_data.shape)

In [ ]:
# Informations générales
print(housing_data.info())
print('
Valeurs manquantes :')
print(housing_data.isnull().sum())

In [ ]:
# Statistiques descriptives
print(housing_data.describe())

print('
Minimums :')
print(housing_data.min())

print('
Maximums :')
print(housing_data.max())

In [ ]:
# Distribution de la variable cible
plt.figure(figsize=(8,5))
sns.histplot(housing_data['medv'], bins=30, kde=True)
plt.title('Distribution de MEDV')
plt.show()

In [ ]:
# Corrélations
corr_matrix=housing_data.corr(numeric_only=True)
plt.figure(figsize=(12,8))
sns.heatmap(corr_matrix,cmap='coolwarm',annot=True,fmt='.2f')
plt.title('Matrice de corrélation')
plt.show()

print(corr_matrix['medv'].sort_values(ascending=False))

In [ ]:
# Relation RM vs MEDV
plt.figure(figsize=(8,6))
sns.scatterplot(data=housing_data,x='rm',y='medv')
plt.title('RM vs MEDV')
plt.show()

In [ ]:
# Relation LSTAT vs MEDV
plt.figure(figsize=(8,6))
sns.scatterplot(data=housing_data,x='lstat',y='medv')
plt.title('LSTAT vs MEDV')
plt.show()

In [ ]:
# Top corrélations avec MEDV
corr_medv=housing_data.corr(numeric_only=True)['medv'].abs().sort_values(ascending=False)
plt.figure(figsize=(8,5))
corr_medv.head(10).plot(kind='bar')
plt.title('Top corrélations avec MEDV')
plt.show()

In [ ]:
# Préparation des données
y=housing_data['medv']
X=housing_data.drop('medv',axis=1)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
# Entraînement XGBoost
model=XGBRegressor(n_estimators=300,learning_rate=0.05,max_depth=4,subsample=0.8,colsample_bytree=0.8,random_state=42)
model.fit(X_train,y_train)

In [ ]:
# Prédictions et métriques
y_pred=model.predict(X_test)
mae=mean_absolute_error(y_test,y_pred)
mse=mean_squared_error(y_test,y_pred)
rmse=mse**0.5
r2=r2_score(y_test,y_pred)
print(f'MAE : {mae:.3f}')
print(f'MSE : {mse:.3f}')
print(f'RMSE : {rmse:.3f}')
print(f'R2 : {r2:.3f}')

In [ ]:
# Comparaison réel vs prédit
plt.figure(figsize=(8,6))
plt.scatter(y_test,y_pred,alpha=0.7)
plt.plot([y_test.min(),y_test.max()],[y_test.min(),y_test.max()],color='red')
plt.xlabel('Valeurs réelles')
plt.ylabel('Valeurs prédites')
plt.title('Réel vs Prédit')
plt.show()

In [ ]:
# Distribution des erreurs
errors=y_test-y_pred
plt.figure(figsize=(8,5))
sns.histplot(errors,bins=25,kde=True)
plt.title('Distribution des erreurs')
plt.show()

In [ ]:
# Importance des variables
feature_importance=pd.DataFrame({'Variable':X.columns,'Importance':model.feature_importances_}).sort_values('Importance',ascending=False)
print(feature_importance)
plt.figure(figsize=(10,6))
sns.barplot(data=feature_importance.head(10),x='Importance',y='Variable')
plt.title('Top 10 variables importantes selon XGBoost')
plt.show()

In [ ]:
# Exemples de prédictions
results=pd.DataFrame({'Valeur réelle':y_test.values,'Valeur prédite':y_pred})
print(results.head(10))